In [0]:
%uv pip install databricks-vectorsearch databricks-langchain

In [0]:
dbutils.library.restartPython()

In [0]:

# Cell — connect to the demo index (creates endpoint + index if missing)
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient()

DEMO_INDEX = "rag_based_chatbot.default.chatbot_idx_demo_v2"
VS_ENDPOINT = "chatbot_endpoint"  # ensure this endpoint name is correct and exists in your environment

# Create endpoint if missing
try:
    vsc.get_endpoint(name=VS_ENDPOINT)
except Exception:
    print(f"Creating endpoint '{VS_ENDPOINT}'...")
    vsc.create_endpoint(name=VS_ENDPOINT)
    for _i in range(60):
        if vsc.get_endpoint(name=VS_ENDPOINT).get("endpoint_status", {}).get("state") == "ONLINE":
            print(f"  Endpoint ready ({_i*10}s)")
            break
        time.sleep(10)

# Create index if missing
try:
    vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=DEMO_INDEX)
except Exception:
    print(f"Creating index '{DEMO_INDEX}' on rag_documents_demo (3K rows)...")
    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=DEMO_INDEX,
        primary_key="doc_id",
        source_table_name="rag_based_chatbot.default.rag_documents_demo",
        pipeline_type="TRIGGERED",
        embedding_source_column="document_text",
        embedding_model_endpoint_name="databricks-bge-large-en",
    )
    for _i in range(120):
        _idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=DEMO_INDEX)
        _st = _idx.describe().get("status", {}).get("detailed_state", "UNKNOWN")
        if _st in ("ONLINE", "ONLINE_NO_PENDING_UPDATE"):
            print(f"  Index ready ({_i*10}s)")
            break
        if _st == "OFFLINE_FAILED":
            raise RuntimeError(f"Index creation failed: {_idx.describe().get('status', {}).get('message', '')}")
        time.sleep(10)

vs_index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=DEMO_INDEX)

In [0]:
# Cell — wait for THIS small index (should take a few minutes, not hours)
import time
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=DEMO_INDEX)

for i in range(60):
    state = index.describe().get("status", {}).get("detailed_state", "UNKNOWN")
    print(f"[demo index] {state} ({i*10}s elapsed)")
    if state in ("ONLINE", "ONLINE_NO_PENDING_UPDATE"):
        break
    time.sleep(10)

In [0]:
# Cell 1 — config
model = "databricks-meta-llama-3-3-70b-instruct"
vector_search_endpoint = "chatbot_endpoint"                          # confirmed
catalog = "rag_based_chatbot"
db = "default"
vector_search_index = "chatbot_idx_demo"                             # ← the fast demo index

chain_config = {
    "llm_model_serving_endpoint_name": model,
    "vector_search_endpoint_name": vector_search_endpoint,
    "vector_search_index": f"{catalog}.{db}.{vector_search_index}",
    "llm_prompt_template": """You are a shopping assistant for baby products, helping customers
understand what other buyers experienced before they purchase. Every passage of context below
comes from a real customer review and includes: the product ID, the star rating, a sentiment
emoji (😍😊😐😞😡) that was computed from that review's own text — NOT your interpretation — and
an excerpt of the review itself.

Rules:
- Base your answer ONLY on the passages in the context. Do not invent product details or reviews.
- Always report sentiment using the emoji already attached to each passage — never assign your own.
- If the passages disagree (e.g. mixed sentiment), say so explicitly rather than picking one side.
- If the context does not contain enough information to answer, say "I don't have enough reviews
  to answer that" rather than guessing.

Context:
{context}""",
}

In [0]:
# Cell  — retriever
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow

mlflow.langchain.autolog()
model_config = mlflow.models.ModelConfig(development_config=chain_config)

# chatbot_idx_demo is orphaned (its VS endpoint was deleted); use the recreated demo index
vector_search_as_retriever = DatabricksVectorSearch(
    index_name="rag_based_chatbot.default.chatbot_idx_demo_v4",
    columns=["doc_id", "asin", "overall", "summary_emoji", "document_text"],
).as_retriever(search_kwargs={"k": 5})

def format_context(docs):
    return "".join(
        f"Passage [{d.metadata.get('summary_emoji', '❓')} {d.metadata.get('overall', '?')}/5, "
        f"product {d.metadata.get('asin', 'unknown')}]: {d.page_content}\n"
        for d in docs
    )

# sanity check — should return real passages now
test = (vector_search_as_retriever | RunnableLambda(format_context) | StrOutputParser()).invoke(
    "Are baby monitors reliable and safe to use?"
)
print(test)

In [0]:
# Cell 3 — prompt + model
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages([
    ("system", model_config.get("llm_prompt_template")),
    ("user", "{question}"),
])
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500},
)

In [0]:
# Cell 4 — full chain, THIS is your live demo
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt | model | StrOutputParser()
)

# Run this live in front of your audience:
answer = chain.invoke({"messages": [{"role": "user", "content": "Are baby monitors reliable and safe to use?"}]})
print(answer)

In [0]:
# Cell — install Gradio (fast, no restart needed)
%pip install gradio -q

In [0]:
# Cell — chat UI wrapping your existing `chain`
# WHY this instead of another Model Serving deploy: `chain` already proved it
# works (you saw a real, cited answer from it earlier). Wrapping it directly
# skips endpoint creation entirely, so there's nothing to get stuck in
# "updating" or "container creation pending" — it just runs.
import gradio as gr

def respond(message, history):
    result = chain.invoke({"messages": [{"role": "user", "content": message}]})
    return result

demo = gr.ChatInterface(
    fn=respond,
    title="chatbot — Baby Products Review Assistant",
    description="Ask about any baby product — answers are grounded in real customer reviews, with sentiment emoji on each source.",
    examples=[
        "Are baby monitors reliable and safe to use?",
        "Do parents complain about strollers breaking?",
        "Is the diaper bag durable and roomy?",
    ],
)

demo.launch(share=True)   # share=True prints a public https://xxxxx.gradio.live link below this cell

In [0]:
demo.close()


In [0]:
# Cellule — version avec erreur visible directement dans le chat
import gradio as gr
import traceback

def respond(message, history):
    try:
        result = chain.invoke({"messages": [{"role": "user", "content": message}]})
        return result
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)  # visible aussi dans la sortie de la cellule du notebook
        # on renvoie l'erreur réelle dans le chat pour la voir tout de suite,
        # sans devoir chercher dans les logs pendant la présentation
        return f"⚠️ Erreur réelle : {type(e).__name__}: {e}"

demo = gr.ChatInterface(
    fn=respond,
    title="chatbot — Baby Products Review Assistant",
    description="Ask about any baby product — answers are grounded in real customer reviews.",
)

demo.launch(share=True)